# Análise Comercial — Gold Layer Salutar Saúde

Análise exploratória das tabelas Gold do projeto **Salutar Saúde**, respondendo a cinco perguntas de negócio chave.

**Âncora temporal:** `hoje = 2025-06-30`

Todos os filtros de data utilizam essa âncora como referência para definir os períodos de análise:
- Últimos 12 meses: `2024-07-01` → `2025-06-30`
- Últimos 6 meses: `2025-01-01` → `2025-06-30`
- Último trimestre: `2025-04-01` → `2025-06-30`
- Próximos 30 dias (vencimento): `2025-07-01` → `2025-07-30`

> **Catálogo/schema:** `homologacao.salutar_saude`

## Q1 — Planos Mais Vendidos (Últimos 12 Meses)

Ranking dos planos por quantidade de contratos e vidas cobertas entre **2024-07-01** e **2025-06-30**.
Percentuais calculados via *window function* (`SUM(...) OVER ()`), cruzando `fat_contratos → dim_plano → dim_operadora`.

In [0]:
%sql
-- anchor: hoje = 2025-06-30
-- Q1: Planos mais vendidos nos últimos 12 meses (data_venda entre 2024-07-01 e 2025-06-30)
-- Percentuais calculados com SUM(...) OVER () para evitar subconsultas extras

SELECT
    p.plano_nome,
    o.operadora_nome,
    p.segmentacao,
    p.acomodacao,
    COUNT(f.contrato_id)                                                                     AS qtd_contratos,
    SUM(f.num_vidas)                                                                         AS vidas_cobertas,
    ROUND(
        COUNT(f.contrato_id) * 100.0 / SUM(COUNT(f.contrato_id)) OVER ()
    , 2)                                                                                     AS pct_contratos,
    ROUND(
        SUM(f.num_vidas) * 100.0 / SUM(SUM(f.num_vidas)) OVER ()
    , 2)                                                                                     AS pct_vidas

FROM homologacao.salutar_saude.gold_fat_contratos  f
JOIN homologacao.salutar_saude.gold_dim_plano       p  ON f.plano_id     = p.plano_id
JOIN homologacao.salutar_saude.gold_dim_operadora   o  ON p.operadora_id = o.operadora_id

WHERE f.data_venda BETWEEN DATE'2024-07-01' AND DATE'2025-06-30'

GROUP BY
    p.plano_nome,
    o.operadora_nome,
    p.segmentacao,
    p.acomodacao

ORDER BY qtd_contratos DESC

plano_nome,operadora_nome,segmentacao,acomodacao,qtd_contratos,vidas_cobertas,pct_contratos,pct_vidas
Plano Flex 529,Vitamed,Ambulatorial,Enfermaria,5,1400,12.50,9.17
Plano Master 758,NovaClin Intermédica,Hospitalar,Apartamento,3,940,7.50,6.16
Plano Premium 384,Vitamed,Ambulatorial + Hospitalar,Enfermaria,3,1496,7.50,9.80
Plano Clássico 621,NovaClin Intermédica,Hospitalar + Obstetrícia,Apartamento,3,826,7.50,5.41
Plano Essencial 423,SulVida Seguros,Hospitalar,Apartamento,2,950,5.00,6.22
Plano Premium 345,Unicare Coop,Ambulatorial,Enfermaria,2,793,5.00,5.19
Plano Clássico 330,PlenaCare,Ambulatorial + Hospitalar,Apartamento,2,760,5.00,4.98
Plano Clássico 338,Vitamed,Ambulatorial,Apartamento,2,951,5.00,6.23
Plano Flex 369,NovaClin Intermédica,Ambulatorial + Hospitalar,Enfermaria,2,808,5.00,5.29
Plano Premium 452,Vitamed,Ambulatorial + Hospitalar,Enfermaria,2,362,5.00,2.37


## Q2 — Empresas com Maior Prêmio Mensal (Últimos 6 Meses)

Top 15 empresas por receita mensal acumulada entre **2025-01-01** e **2025-06-30**.
Ranking via `RANK() OVER (ORDER BY premio_mensal_total DESC)`, cruzando `fat_contratos → dim_empresa`.

In [0]:
%sql
-- anchor: hoje = 2025-06-30
-- Q2: Top 15 empresas por prêmio mensal total — últimos 6 meses (2025-01-01 a 2025-06-30)
-- Rank calculado com RANK() OVER para preservar empates

SELECT
    e.empresa_nome,
    e.setor,
    e.porte,
    e.uf,
    COUNT(f.contrato_id)                                              AS qtd_contratos,
    SUM(f.num_vidas)                                                  AS vidas_total,
    ROUND(SUM(f.receita_total_mensal), 2)                             AS premio_mensal_total,
    RANK() OVER (ORDER BY SUM(f.receita_total_mensal) DESC)           AS rank

FROM homologacao.salutar_saude.gold_fat_contratos f
JOIN homologacao.salutar_saude.gold_dim_empresa   e  ON f.empresa_id = e.empresa_id

WHERE f.data_venda BETWEEN DATE'2025-01-01' AND DATE'2025-06-30'

GROUP BY
    e.empresa_nome,
    e.setor,
    e.porte,
    e.uf

ORDER BY premio_mensal_total DESC
LIMIT 15

empresa_nome,setor,porte,uf,qtd_contratos,vidas_total,premio_mensal_total,rank
Empresa Q17 Ltda,Logística,Grande,CE,1,626,404725526.40,1
Empresa D04 Ltda,Indústria,Média,MG,1,705,367393773.60,2
Empresa N14 Ltda,Educação,Pequena,DF,2,910,182781269.45,3
Empresa Y25 Ltda,Educação,Média,GO,1,584,139922615.68,4
Empresa T20 Ltda,Saúde,Grande,PR,1,434,132344637.04,5
Empresa K11 Ltda,Saúde,Pequena,PE,1,486,97343792.82,6
Empresa J10 Ltda,Agronegócio,Pequena,RJ,1,623,72868073.60,7
Empresa M13 Ltda,Educação,Grande,RJ,1,391,53752326.18,8
Empresa Z26 Ltda,Agronegócio,Grande,DF,2,379,48460980.68,9
Empresa M39 Ltda,Varejo,Pequena,SC,1,247,47591834.03,10


## Q3 — Performance dos Corretores (Último Trimestre)

Vendas entre **2025-04-01** e **2025-06-30**, cruzando `fat_contratos → dim_corretor`.

**Normalização justa por dias ativos:** corretores admitidos no meio do trimestre tiveram menos dias disponíveis para vender. Para uma comparação equitativa, calculamos:

```
dias_ativos_no_trimestre = LEAST(DATEDIFF(DATE'2025-06-30', data_admissao) + 1, 91)
```

Onde **91** é o total de dias do trimestre (2025-04-01 a 2025-06-30). As métricas `valor_por_dia_ativo` e `vidas_por_dia_ativo` eliminam a vantagem de quem estava na equipe desde antes do período, permitindo uma comparação justa entre todos os corretores.

In [0]:
%sql
-- anchor: hoje = 2025-06-30
-- Q3: Performance dos corretores no último trimestre (2025-04-01 a 2025-06-30)
-- Normalização justa: dias_ativos_no_trimestre = LEAST(DATEDIFF('2025-06-30', data_admissao) + 1, 91)
-- Métricas comparativas: valor_por_dia_ativo e vidas_por_dia_ativo

WITH vendas AS (
    SELECT
        f.corretor_id,
        COUNT(f.contrato_id)          AS qtd_contratos,
        SUM(f.valor_mensal)           AS total_valor_vendido,
        SUM(f.num_vidas)              AS total_vidas
    FROM homologacao.salutar_saude.gold_fat_contratos f
    WHERE f.data_venda BETWEEN DATE'2025-04-01' AND DATE'2025-06-30'
    GROUP BY f.corretor_id
)

SELECT
    c.corretor_nome,
    c.regiao,
    c.senioridade,
    v.qtd_contratos,
    ROUND(v.total_valor_vendido, 2)                                                                                        AS total_valor_vendido,
    v.total_vidas,
    -- Dias disponíveis para vender no trimestre (máx. 91 dias = 2025-04-01 a 2025-06-30)
    LEAST(DATEDIFF(DATE'2025-06-30', c.data_admissao) + 1, 91)                                                            AS dias_ativos_no_trimestre,
    -- Métricas normalizadas: base para comparação justa entre corretores
    ROUND(
        v.total_valor_vendido / NULLIF(LEAST(DATEDIFF(DATE'2025-06-30', c.data_admissao) + 1, 91), 0)
    , 2)                                                                                                                   AS valor_por_dia_ativo,
    ROUND(
        v.total_vidas / NULLIF(LEAST(DATEDIFF(DATE'2025-06-30', c.data_admissao) + 1, 91), 0)
    , 4)                                                                                                                   AS vidas_por_dia_ativo,
    -- Ticket médio por contrato (independente da normalização)
    ROUND(v.total_valor_vendido / NULLIF(v.qtd_contratos, 0), 2)                                                          AS valor_por_contrato,
    ROUND(CAST(v.total_vidas AS DOUBLE) / NULLIF(v.qtd_contratos, 0), 2)                                                  AS vidas_por_contrato

FROM vendas                                       v
JOIN homologacao.salutar_saude.gold_dim_corretor  c  ON v.corretor_id = c.corretor_id

ORDER BY valor_por_dia_ativo DESC

corretor_nome,regiao,senioridade,qtd_contratos,total_valor_vendido,total_vidas,dias_ativos_no_trimestre,valor_por_dia_ativo,vidas_por_dia_ativo,valor_por_contrato,vidas_por_contrato
Carla Almeida,Norte,Júnior,1,239593.52,584,91,2632.90,6.4176,239593.52,584.0
Ana Rodrigues,Centro-Oeste,Pleno,1,185810.64,254,91,2041.88,2.7912,185810.64,254.0
Carla Rocha,Sul,Pleno,1,169489.15,254,91,1862.52,2.7912,169489.15,254.0
Rafael Silva,Sul,Júnior,1,75218.43,328,91,826.58,3.6044,75218.43,328.0
Tiago Nunes,Norte,Pleno,1,40015.85,78,91,439.73,0.8571,40015.85,78.0


## Q4 — Contratos a Expirar nos Próximos 30 Dias

Contratos com `status = 'Ativo'` e `vigencia_fim` entre **2025-07-01** e **2025-07-30**.

**Valor em risco** = `receita_total_mensal` — a receita mensal que seria perdida caso o contrato não seja renovado. Útil para priorizar ações de retenção pelo time comercial, ordenando por data de expiração e valor em risco.

In [0]:
%sql
-- anchor: hoje = 2025-06-30
-- Q4: Contratos ativos a expirar nos próximos 30 dias (vigencia_fim entre 2025-07-01 e 2025-07-30)
-- valor_em_risco = receita_total_mensal (receita mensal perdida se não renovado)
-- Cruzamento: fat_contratos → dim_empresa → dim_plano → dim_corretor

SELECT
    f.contrato_id,
    e.empresa_nome,
    p.plano_nome,
    c.corretor_nome,
    f.vigencia_fim,
    f.num_vidas,
    ROUND(f.receita_total_mensal, 2)                    AS valor_em_risco,
    DATEDIFF(f.vigencia_fim, DATE'2025-06-30')          AS dias_para_expirar

FROM homologacao.salutar_saude.gold_fat_contratos  f
JOIN homologacao.salutar_saude.gold_dim_empresa    e  ON f.empresa_id  = e.empresa_id
JOIN homologacao.salutar_saude.gold_dim_plano      p  ON f.plano_id    = p.plano_id
JOIN homologacao.salutar_saude.gold_dim_corretor   c  ON f.corretor_id = c.corretor_id

WHERE f.vigencia_fim BETWEEN DATE'2025-07-01' AND DATE'2025-07-30'
  AND f.status = 'Ativo'

ORDER BY
    f.vigencia_fim   ASC,
    valor_em_risco   DESC

contrato_id,empresa_nome,plano_nome,corretor_nome,vigencia_fim,num_vidas,valor_em_risco,dias_para_expirar
4,Empresa A01 Ltda,Plano Premium 452,Igor Oliveira,2025-07-06,408,80829410.40,6
11,Empresa E05 Ltda,Plano Premium 743,Eduardo Pereira,2025-07-10,635,106021244.65,10
59,Empresa W23 Ltda,Plano Flex 369,Igor Oliveira,2025-07-20,469,166047470.82,20
28,Empresa K11 Ltda,Plano Essencial 626,Gabriel Pereira,2025-07-24,470,162996319.60,24


## Q5 — Sinistralidade por Empresa (Últimos 12 Meses)

Agregação de `gold_fat_sinistralidade` no período **2024-07** a **2025-06**, por empresa.

| Alerta | Critério |
|--------|----------|
| 🔴 PREJUÍZO | Sinistralidade acumulada > 100% |
| 🟡 ATENÇÃO | Sinistralidade acumulada > 80% |
| 🟢 SAUDÁVEL | Sinistralidade acumulada ≤ 80% |

`sinistralidade_acumulada_pct` é calculada a partir dos valores absolutos acumulados (custo / receita), não pela média das taxas mensais — o que representa com mais fidelidade o resultado financeiro real do período.

> ⚠️ **Empresa I09 Ltda** possui `receita_premios = 0` no período inteiro (sinistros sem receita correspondente). Como a divisão por zero é matematicamente indefinida, ela recebe o valor sentinela **9.999,99%** e alerta `🔴 SEM RECEITA` para ser incluída no gráfico e priorizada na investigação.

In [0]:
%sql
-- anchor: hoje = 2025-06-30
-- Q5: Sinistralidade por empresa — últimos 12 meses (ano_mes entre '2024-07' e '2025-06')
-- Fonte: gold_fat_sinistralidade (grain: empresa × ano_mes)
-- CTE base: calcula sinistralidade_acumulada_pct uma única vez para reutilizar no CASE do alerta
-- Tratamento I09: receita = 0 → divisão indefinida → valor sentinela 9999.99 + alerta '🔴 SEM RECEITA'

WITH base AS (
    SELECT
        empresa_nome,
        ROUND(SUM(receita_premios), 2)                                                AS receita_acumulada,
        ROUND(SUM(custo_sinistros), 2)                                                AS custo_acumulado,
        ROUND(SUM(margem_tecnica),  2)                                                AS margem_acumulada,
        -- Média das taxas mensais (referência complementar; NULL quando receita mensal = 0)
        ROUND(AVG(sinistralidade_pct), 2)                                             AS sinistralidade_media_pct,
        -- Taxa acumulada real; sentinela 9999.99 quando receita total = 0 e há custo
        CASE
            WHEN SUM(receita_premios) = 0 AND SUM(custo_sinistros) > 0 THEN 9999.99
            ELSE ROUND(SUM(custo_sinistros) / NULLIF(SUM(receita_premios), 0) * 100, 2)
        END                                                                           AS sinistralidade_acumulada_pct,
        SUM(CASE WHEN margem_tecnica < 0 THEN 1 ELSE 0 END)                          AS meses_com_prejuizo
    FROM homologacao.salutar_saude.gold_fat_sinistralidade
    WHERE ano_mes BETWEEN '2024-07' AND '2025-06'
    GROUP BY empresa_nome
)

SELECT
    empresa_nome,
    receita_acumulada,
    custo_acumulado,
    margem_acumulada,
    sinistralidade_media_pct,
    sinistralidade_acumulada_pct,
    meses_com_prejuizo,
    -- Semáforo: sentinela → SEM RECEITA | > 100% → PREJUÍZO | > 80% → ATENÇÃO | demais → SAUDÁVEL
    CASE
        WHEN sinistralidade_acumulada_pct = 9999.99 THEN '🔴 SEM RECEITA'
        WHEN sinistralidade_acumulada_pct > 100     THEN '🔴 PREJUÍZO'
        WHEN sinistralidade_acumulada_pct > 80      THEN '🟡 ATENÇÃO'
        ELSE                                             '🟢 SAUDÁVEL'
    END                                                                               AS alerta

FROM base

ORDER BY sinistralidade_acumulada_pct DESC NULLS LAST

empresa_nome,receita_acumulada,custo_acumulado,margem_acumulada,sinistralidade_media_pct,sinistralidade_acumulada_pct,meses_com_prejuizo,alerta
Empresa I09 Ltda,0.00,507259.90,-507259.90,null,9999.99,11,🔴 SEM RECEITA
Empresa K37 Ltda,117474.98,1431446.60,-1313971.62,1218.51,1218.51,6,🔴 PREJUÍZO
Empresa U21 Ltda,452325.16,1884787.43,-1432462.27,181.00,416.69,11,🔴 PREJUÍZO
Empresa G07 Ltda,692335.48,2284979.05,-1592643.57,491.67,330.04,10,🔴 PREJUÍZO
Empresa I35 Ltda,879283.98,2278703.39,-1399419.41,297.11,259.15,9,🔴 PREJUÍZO
Empresa N40 Ltda,577021.44,1427862.60,-850841.16,247.45,247.45,4,🔴 PREJUÍZO
Empresa P16 Ltda,1026028.06,1993877.71,-967849.65,173.95,194.33,11,🔴 PREJUÍZO
Empresa T20 Ltda,2151657.38,4167539.59,-2015882.21,159.80,193.69,8,🔴 PREJUÍZO
Empresa E31 Ltda,2160311.36,3523857.96,-1363546.60,155.82,163.12,9,🔴 PREJUÍZO
Empresa D04 Ltda,2186743.92,2858074.38,-671330.46,501.57,130.70,3,🔴 PREJUÍZO


Databricks visualization. Run in Databricks to view.